# Modelling LSTM - 4-Split (train_core + val + train_full + test)

**Project:** Prediksi PM2.5 di Jakarta - Eksperimen Tambahan

**Tujuan notebook ini:** uji apakah strategi split 4-bagian (train_core + validation + train_full + test) memberi hasil berbeda dibanding 3-split (train + val + test) yang sudah dipakai di notebook utama. Dilakukan pada **dataset v1 (proposal scope) dan v2 (out-of-scope, dataset bersih)**.

**Date boundaries (sesuai setup awal user):**
- Train_core : 2022-01-01 -> 2024-01-15
- Validation : 2024-01-16 -> 2024-05-26
- Train_full : 2022-01-01 -> 2024-05-26 (train_core + val)
- Test       : 2024-05-27 -> 2025-01-01

**Workflow:**
1. Tune hyperparameter pakai **train_core**, evaluasi di **val**
2. Pilih best by **val_RMSE**
3. Retrain final pada **train_full** (train_core + val)
4. Evaluasi sekali pada **test**

**Note:** notebook ini self-contained. Untuk reproducibility, kode utama juga tersedia di `src/eksperimen_4split.py`.

## 1. Setup & Functions

In [1]:
import os, time, warnings, json
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM = 42
np.random.seed(RANDOM)
DIR_OUT = Path('outputs')
TARGET = 'ISPU PM2.5'

# Boundaries 4-split
BATAS_VAL_MULAI  = pd.Timestamp('2024-01-16')
BATAS_VAL_AKHIR  = pd.Timestamp('2024-05-26')
BATAS_TEST_MULAI = pd.Timestamp('2024-05-27')
print('Setup OK')

Setup OK


In [2]:
def load_dan_fe(versi):
    """versi = 'v1' atau 'v2'. Returns df + fitur info."""
    if versi == 'v1':
        path = Path('../data/final for modelling/dataset_final_model.csv')
        kolom_cuaca = ['temp', 'humidity', 'visibility', 'windgust', 'solarenergy', 'precip']
    else:
        path = Path('../data/final for modelling/dataset_final_model_v2.csv')
        kolom_cuaca = ['temp', 'humidity', 'visibility', 'windgust', 'solarenergy', 'precip',
                       'cloudcover', 'tempmax', 'tempmin', 'feelslike', 'uvindex',
                       'precipprob', 'sealevelpressure', 'dew', 'winddir_sin', 'winddir_cos']
    df = pd.read_csv(path)
    df['tanggal'] = pd.to_datetime(df['tanggal'])
    kolom_stasiun = [c for c in df.columns if c.startswith('station_')]
    df['station'] = df[kolom_stasiun].idxmax(axis=1).str.replace('station_', '', regex=False)
    if 'bulan' not in df.columns: df['bulan'] = df['tanggal'].dt.month
    if 'hari_minggu' not in df.columns: df['hari_minggu'] = df['tanggal'].dt.dayofweek
    df['bulan_sin'] = np.sin(2*np.pi*df['bulan']/12); df['bulan_cos'] = np.cos(2*np.pi*df['bulan']/12)
    df['hari_minggu_sin'] = np.sin(2*np.pi*df['hari_minggu']/7); df['hari_minggu_cos'] = np.cos(2*np.pi*df['hari_minggu']/7)
    def musim(b):
        if b in (11,12,1,2,3): return 'Hujan'
        if b == 4: return 'Transisi'
        return 'Kemarau'
    df['musim'] = df['bulan'].map(musim)
    for lag in [1,3,7]: df[f'pm25_lag_{lag}'] = df.groupby('station')[TARGET].shift(lag)
    prev = df.groupby('station')[TARGET].shift(1)
    def rs(s,w,f): return s.groupby(df['station']).rolling(w).agg(f).reset_index(level=0, drop=True)
    df['pm25_rolling_mean_3'] = rs(prev,3,'mean'); df['pm25_rolling_mean_7'] = rs(prev,7,'mean')
    df['pm25_rolling_max_7']  = rs(prev,7,'max');  df['pm25_rolling_std_7']  = rs(prev,7,'std')
    fitur_lag = ['pm25_lag_1','pm25_lag_3','pm25_lag_7','pm25_rolling_mean_3',
                 'pm25_rolling_mean_7','pm25_rolling_max_7','pm25_rolling_std_7']
    kolom_cuaca = [c for c in kolom_cuaca if c in df.columns]
    df = df.dropna(subset=fitur_lag + [TARGET]).reset_index(drop=True)
    return df, kolom_cuaca, fitur_lag, kolom_stasiun

def split_4(df):
    return (df['tanggal'] < BATAS_VAL_MULAI,
            (df['tanggal'] >= BATAS_VAL_MULAI) & (df['tanggal'] <= BATAS_VAL_AKHIR),
            df['tanggal'] <= BATAS_VAL_AKHIR,
            df['tanggal'] >= BATAS_TEST_MULAI)

def ev(y, yp):
    return {'MAE': float(mean_absolute_error(y,yp)),
            'RMSE': float(np.sqrt(mean_squared_error(y,yp))),
            'R2': float(r2_score(y,yp))}

print('Functions ready')

Functions ready


In [3]:
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
tf.random.set_seed(RANDOM); keras.utils.set_random_seed(RANDOM)

def arsitektur_A(L, F):
    m = keras.Sequential([layers.Input(shape=(L,F)), layers.LSTM(64), layers.Dropout(0.2),
                          layers.Dense(32, activation='relu'), layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m
def arsitektur_B(L, F):
    m = keras.Sequential([layers.Input(shape=(L,F)), layers.LSTM(128), layers.Dropout(0.3),
                          layers.Dense(64, activation='relu'), layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m
def arsitektur_C(L, F):
    m = keras.Sequential([layers.Input(shape=(L,F)), layers.LSTM(64, return_sequences=True),
                          layers.Dropout(0.2), layers.LSTM(32), layers.Dropout(0.2),
                          layers.Dense(32, activation='relu'), layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
    return m

def run_lstm_4split(df, kolom_cuaca, fitur_lag, kolom_stasiun, label, lookback=14):
    musim_dummies = pd.get_dummies(df['musim'], prefix='musim').astype(float)
    for col in ['musim_Hujan','musim_Kemarau','musim_Transisi']:
        if col not in musim_dummies.columns: musim_dummies[col] = 0.0
    for c in ['musim_Hujan','musim_Kemarau','musim_Transisi']:
        df[c] = musim_dummies[c].values
    musim_cols = ['musim_Hujan','musim_Kemarau','musim_Transisi']

    for c in kolom_cuaca:
        df[c] = df[c].fillna(df[c].median())

    FITUR = ([TARGET] + kolom_cuaca + ['bulan_sin','bulan_cos','hari_minggu_sin','hari_minggu_cos']
             + musim_cols + kolom_stasiun)
    F = len(FITUR)

    Xs, ys, tgs, sts = [], [], [], []
    for st, sub in df.groupby('station'):
        sub = sub.sort_values('tanggal').reset_index(drop=True)
        if len(sub) <= lookback: continue
        arr = sub[FITUR].values.astype(np.float32); y = sub[TARGET].values.astype(np.float32); tgl = sub['tanggal'].values
        for i in range(lookback, len(sub)):
            Xs.append(arr[i-lookback:i]); ys.append(y[i]); tgs.append(tgl[i]); sts.append(st)
    X = np.stack(Xs); y = np.array(ys, dtype=np.float32); tgl = pd.to_datetime(np.array(tgs))

    m_core = tgl < BATAS_VAL_MULAI
    m_val  = (tgl >= BATAS_VAL_MULAI) & (tgl <= BATAS_VAL_AKHIR)
    m_full = tgl <= BATAS_VAL_AKHIR
    m_test = tgl >= BATAS_TEST_MULAI
    print(f'  Train_core {m_core.sum()} | Val {m_val.sum()} | Train_full {m_full.sum()} | Test {m_test.sum()}')

    sx_tune = StandardScaler().fit(X[m_core].reshape(-1, F))
    def trans(a, sx): return sx.transform(a.reshape(-1, F)).reshape(a.shape).astype(np.float32)
    X_core_s, X_val_s = trans(X[m_core], sx_tune), trans(X[m_val], sx_tune)
    y_core_a, y_val_a = y[m_core], y[m_val]

    hasil_arch = []
    for nm, fac in [('A_LSTM64_Dropout0.2_Dense32', arsitektur_A),
                    ('B_LSTM128_Dropout0.3_Dense64', arsitektur_B),
                    ('C_LSTM64+LSTM32_Dropout0.2_Dense32', arsitektur_C)]:
        keras.utils.set_random_seed(RANDOM)
        model = fac(lookback, F)
        cb = [keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor='val_loss', verbose=1),
              keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5, monitor='val_loss', verbose=1)]
        h = model.fit(X_core_s, y_core_a, validation_data=(X_val_s, y_val_a),
                      epochs=80, batch_size=128, verbose=1, callbacks=cb)
        val_rmse = float(np.sqrt(mean_squared_error(y_val_a, model.predict(X_val_s, verbose=0).flatten())))
        print(f'  Arch {nm}: val_RMSE {val_rmse:.4f}')
        hasil_arch.append((nm, fac, val_rmse))

    nm_best, fac_best, val_rmse = min(hasil_arch, key=lambda x: x[2])
    print(f'  -> Best arch: {nm_best}')

    sx_final = StandardScaler().fit(X[m_full].reshape(-1, F))
    X_full_s, X_test_s = trans(X[m_full], sx_final), trans(X[m_test], sx_final)
    y_full_a, y_test_a = y[m_full], y[m_test]
    keras.utils.set_random_seed(RANDOM)
    model_final = fac_best(lookback, F)
    model_final.fit(X_full_s, y_full_a, epochs=40, batch_size=128, verbose=1)
    metric_test = ev(y_test_a, model_final.predict(X_test_s, verbose=0).flatten())
    print(f'  Test: RMSE {metric_test["RMSE"]:.4f} | R2 {metric_test["R2"]:.4f} | MAE {metric_test["MAE"]:.4f}')
    return {'model':'LSTM','dataset':label,'tuning': f'Arch Search @ L{lookback}',
            'val_RMSE': round(val_rmse,4), 'test_RMSE': round(metric_test['RMSE'],4),
            'test_R2': round(metric_test['R2'],4), 'test_MAE': round(metric_test['MAE'],4),
            'params': {'arch': nm_best, 'lookback': lookback}}

print('LSTM function ready')

LSTM function ready


## 2. LSTM pada Dataset v1

In [4]:
df_v1, cuaca_v1, fitur_lag, kolom_stasiun = load_dan_fe('v1')
hasil_v1 = run_lstm_4split(df_v1, cuaca_v1, fitur_lag, kolom_stasiun, 'v1 (6 cuaca)')

  Train_core 5068 | Val 924 | Train_full 5992 | Test 1540
Epoch 1/80
40/40 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - loss: 5076.4463 - mae: 68.1117 - val_loss: 3780.7993 - val_mae: 59.0317 - learning_rate: 0.0010
Epoch 2/80
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 2014.0227 - mae: 39.7050 - val_loss: 601.8525 - val_mae: 20.2963 - learning_rate: 0.0010
Epoch 3/80
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 578.0335 - mae: 19.1929 - val_loss: 363.7730 - val_mae: 15.0653 - learning_rate: 0.0010
Epoch 4/80
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 473.5041 - mae: 16.6282 - val_loss: 357.1210 - val_mae: 14.8943 - learning_rate: 0.0010
Epoch 5/80
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 459.4546 - mae: 16.3886 - val_loss: 326.0340 - val_mae: 14.2092 - learning_rate: 0.0010
Epoch 6/80
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 412.6917 - mae: 15.4351 - val_loss: 267.6922 - val_mae: 12.8908 - learning_rate: 0.0010
Epoch 7/80
40/40 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss

## 3. LSTM pada Dataset v2

In [5]:
df_v2, cuaca_v2, fitur_lag, kolom_stasiun = load_dan_fe('v2')
hasil_v2 = run_lstm_4split(df_v2, cuaca_v2, fitur_lag, kolom_stasiun, 'v2 (15 cuaca)')

  Train_core 3581 | Val 727 | Train_full 4308 | Test 1154
Epoch 1/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - loss: 5934.1084 - mae: 74.0507 - val_loss: 4508.1226 - val_mae: 63.9242 - learning_rate: 0.0010
Epoch 2/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 4202.0801 - mae: 61.1831 - val_loss: 1737.3444 - val_mae: 36.7740 - learning_rate: 0.0010
Epoch 3/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 1602.3007 - mae: 34.5439 - val_loss: 604.6122 - val_mae: 19.8269 - learning_rate: 0.0010
Epoch 4/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 678.4899 - mae: 20.7362 - val_loss: 441.4426 - val_mae: 16.9564 - learning_rate: 0.0010
Epoch 5/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 487.4311 - mae: 16.9836 - val_loss: 500.1034 - val_mae: 18.1261 - learning_rate: 0.0010
Epoch 6/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 478.0443 - mae: 16.6732 - val_loss: 490.3402 - val_mae: 17.9217 - learning_rate: 0.0010
Epoch 7/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - lo

## 4. Perbandingan v1 vs v2

In [6]:
df_perbandingan = pd.DataFrame([hasil_v1, hasil_v2])
df_perbandingan.to_csv(DIR_OUT/'lstm_4split_v1_v2.csv', index=False)
display(df_perbandingan[['model','dataset','tuning','val_RMSE','test_RMSE','test_R2','test_MAE']])

,model,dataset,tuning,val_RMSE,test_RMSE,test_R2,test_MAE
0,LSTM,v1 (6 cuaca),Arch Search @ L14,11.7219,13.3192,0.5857,9.6633
1,LSTM,v2 (15 cuaca),Arch Search @ L14,14.0481,16.4157,0.4632,12.2286


## 5. Interpretasi

- LSTM butuh banyak data sequence - v2 yang lebih kecil (training_core ~3580) jelas rugi.
- Arch B (LSTM128) biasanya menang di v1 (data cukup), arch A (LSTM64) menang di v2 (data sedikit, simpler model overfit lebih kecil).
- LSTM secara umum kalah R^2 dari RF/XGBoost di dataset ini.

**File:** `lstm_4split_v1_v2.csv`.